# Your first reservoir simulation with kusim

In this tutorial you will **build** a small oil reservoir model, **run** a waterflood on it, and **look at the results** in
four ways:

1. **Line plots**: what the wells do through time (rates, pressures, water cut, cumulative volumes).
2. **3-D data**: pressure and water saturation in every block, at one time, and how they **change** over time.
3. **Maps of a single layer**, side by side at several times.
4. **Movies**: a time-lapse of one layer, of the whole 3-D model, and of the **average map** (all layers averaged into one map).

You need Python, and kusim installed (`py -m pip install -e .[dev]` from the repository folder). No simulation knowledge is assumed;
each step says what it is for. Run the cells from top to bottom (**Kernel → Restart & Run All** runs everything, about 30 seconds).

**Units.** kusim uses oilfield units everywhere: lengths in **ft**, pressure in **psi**, permeability in **md**, rates in **STB/day**
(stock-tank barrels per day), time in **days**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image

from kusim import Simulator, viz
from kusim.grid import Grid
from kusim.fluids import PVTTable
from kusim.cases import corey_rock_fluid
from kusim.model import ReservoirModel, UniformStart
from kusim.wells import Injector, Producer
from kusim.checks import check_model

## 1. Build the model

A reservoir model is a handful of ingredients: the **grid** (the shape), the **rock** (permeability, porosity), the **fluids**
(PVT and relative permeability), the **initial state**, the **wells**, and the **time** to simulate. We will make each one in turn
and put them together at the end.

### 1.1 The grid

The reservoir is cut into blocks: here **20 blocks in x, 10 in y and 3 layers in z**. Each block is 100 ft × 100 ft; the three
layers are 15, 25 and 20 ft thick. The top of the reservoir is 7000 ft deep.

Blocks are numbered **from 1**, along x first, then y, then down the layers: block 1 is the top layer at `(i=1, j=1)`, and the
last block is the bottom layer at `(i=20, j=10)`.

In [ ]:
nx, ny, nz = 20, 10, 3
grid = Grid(dx=[100.0] * nx, dy=[100.0] * ny, dz=[15.0, 25.0, 20.0], top_depth=7000.0)

print(f"{grid.nx} x {grid.ny} x {grid.nz} = {grid.n_cells} blocks")
print("block number of (i=1, j=1, k=1):  ", grid.block_number(1, 1, 1))
print("block number of (i=2, j=5, k=3):  ", grid.block_number(2, 5, 3))
print("last block (i=20, j=10, k=3):     ", grid.block_number(20, 10, 3))

You never need to count blocks yourself: `grid.block_number(i, j, k)` gives the number, and `grid.column_blocks(i, j)` gives the
blocks of a whole **vertical column** (all layers at one `(i, j)`), which is what a vertical well passes through.

### 1.2 The rock

**Permeability** says how easily fluid flows. We give each layer its own average (the bottom layer is the best, 600 md) and add
some random variation from block to block, so the reservoir is not perfectly uniform. The random numbers use a fixed seed, so
everyone gets the same reservoir. Vertical permeability `kz` is 10 % of the horizontal one, which is typical.

In [ ]:
rng = np.random.default_rng(7)
layer_permeability = np.array([80.0, 250.0, 600.0])                       # md, top layer first
kx = np.repeat(layer_permeability, grid.n_layer) * np.exp(0.4 * rng.standard_normal(grid.n_cells))
ky = kx
kz = 0.1 * kx

fig, axes = plt.subplots(1, nz, figsize=(13, 3.2), sharey=True)
for k, ax in enumerate(axes):
    mesh = ax.pcolormesh(grid.x_edges, grid.y_edges, grid.to_3d(kx)[k], cmap="YlGn", vmin=kx.min(), vmax=kx.max())
    ax.set(title=f"layer {k + 1}: mean {grid.to_3d(kx)[k].mean():.0f} md", xlabel="x (ft)", aspect="equal")
axes[0].set_ylabel("y (ft)")
fig.colorbar(mesh, ax=axes, label="kx (md)")
plt.show()

**Porosity** (the fraction of the rock that is pore space) is 20 % everywhere; we will give a single number.

### 1.3 The fluids

Two tables describe the fluids.

* **PVT** (pressure-volume-temperature): how the oil and water volume and viscosity change with pressure. `bo` and `bw` are the
  formation volume factors (reservoir barrels per stock-tank barrel), `mu_o` and `mu_w` the viscosities in cp.
* **Relative permeability**: how much each fluid can flow when both are present, as a function of water saturation `Sw`. We
  use Corey power laws; the water cannot be pushed below the connate saturation `swc = 0.2`, and oil cannot be pushed below the
  residual saturation `sor = 0.2`.

In [ ]:
pvt = PVTTable(
    pressure=[1000, 2000, 3000, 4000, 5000],           # psi
    bo=[1.300, 1.285, 1.270, 1.255, 1.240],            # rb/STB
    bw=[1.030, 1.027, 1.024, 1.021, 1.018],            # rb/STB
    mu_o=[1.5] * 5,                                    # cp
    mu_w=[0.5] * 5,                                    # cp
)
rock_fluid = corey_rock_fluid(swc=0.2, sor=0.2, krw_max=0.4, nw=2, no=2)

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(rock_fluid.sw, rock_fluid.krw, label="water  krw")
ax.plot(rock_fluid.sw, rock_fluid.kro, label="oil  kro")
ax.set(xlabel="water saturation Sw", ylabel="relative permeability", title="Relative permeability")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

Read the plot like this: at low `Sw` only oil flows; as `Sw` rises, water flows more and oil less. Because the water is less
viscous (0.5 cp against 1.5 cp), it tends to run ahead of the oil, and that is why the **water cut** (the water share of the
produced liquid) rises after the water arrives at the producers.

### 1.4 The initial state

We start with the simplest state: pressure 3000 psi and water saturation 0.2 (connate water only) in every block. (kusim can
also start from capillary-gravity equilibrium, `Equilibrium`; see the docs.)

### 1.5 The wells

One **injector** pushes water into the reservoir at a fixed rate, and two **producers** pull fluids out at a fixed
bottom-hole pressure. All three are vertical and open to all three layers, so we hand each one the blocks of its column.

* `Injector(blocks, "water_rate", 1200)` injects 1200 STB/day.
* `Producer(blocks, "bhp", 2500)` holds the pressure at the bottom of the well at 2500 psi; the rate follows from that.

The injector is at `(i=2, j=5)` on the left, the producers at `(i=19, j=3)` and `(i=19, j=8)` on the right.

In [ ]:
injector = Injector(grid.column_blocks(2, 5), "water_rate", 1200.0, radius=0.3)
producer_1 = Producer(grid.column_blocks(19, 3), "bhp", 2500.0, radius=0.3)
producer_2 = Producer(grid.column_blocks(19, 8), "bhp", 2500.0, radius=0.3)

### 1.6 Put it together

The model gathers everything. We also choose the **time step** `dt = 5` days and the **end time**, 1500 days (about 4 years).
The simulation advances in steps of `dt`; a smaller step is more accurate and slower, and a step that is too large makes this
kind of simulator unstable. The next cell checks that for us.

In [ ]:
model = ReservoirModel(
    grid=grid, kx=kx, ky=ky, kz=kz, porosity=0.2, rock_compressibility=4e-6,
    pvt=pvt, rock_fluid=rock_fluid,
    gamma_o=0.85, gamma_w=1.0,                          # specific gravities (water = 1)
    swc=0.2, sor=0.2,
    initial=UniformStart(pressure=3000.0, sw=0.2),
    dt=5.0, t_end=1500.0,
    injectors=[injector], producers=[producer_1, producer_2],
)

check = check_model(model)
print("runnable:", check["runnable"], "| likely stable:", check["stability"]["likely_stable"],
      "| about", round(check["estimate"]["seconds"], 1), "seconds")

`check_model` is a quick sanity check before a long run: it tells you if the model is runnable, whether the time step looks
safe, and roughly how long the run will take.

## 2. Run it

`Simulator(model).run()` runs the whole simulation and returns a `Results` object. By default only the first and last states are
kept; `save_every=4` keeps the pressure and saturation of **every 4th step** (every 20 days), which is what we need for time
plots and movies. (The well rates are always saved at every step.)

In [ ]:
results = Simulator(model, save_every=4).run()

print(f"saved {len(results.times)} snapshots, from t = {results.times[0]:g} to {results.times[-1]:g} days")
print("results.pressure and results.sw have shape", results.sw.shape, "= (snapshots, blocks)")
if results.stop_reason:
    print("stopped early:", results.stop_reason)

A `Results` object holds:

| What | Where | Shape / type |
|---|---|---|
| times of the saved snapshots | `results.times` | 1-D array (days) |
| oil pressure in every block | `results.pressure` | `(snapshots, blocks)` |
| water saturation in every block | `results.sw` | `(snapshots, blocks)` |
| well histories | `results.wells` | a table (pandas DataFrame), one row per well per step |
| per-perforation rates | `results.perforations` | a table |

The well table is the easiest to start with:

In [ ]:
results.well("P1")[["time", "bhp", "oil_rate", "water_rate", "water_cut", "cum_oil"]].iloc[[0, 59, 119, 179, 239, 299]]

## 3. Line plots: what the wells do

`viz.plot_wells` draws the usual four-panel summary: surface rates, bottom-hole pressure, water cut and cumulative volumes.

In [ ]:
viz.plot_wells(results)
plt.show()

Things to notice: in the first few days the producers give a burst of oil while the pressure falls from 3000 psi to its working level, then the oil rate is steady. The water cut stays at zero until the injected water **breaks through** (around day 800), then climbs, and the oil rate falls. The injector's bottom-hole pressure (about 2800 psi) is what it takes to push 1200 STB/day in.

Plotting your own line plots is just as easy, because the data are a table. Here the two producers are compared, one panel
each for oil rate and water cut:

In [ ]:
fig, (ax_oil, ax_cut) = plt.subplots(1, 2, figsize=(11, 3.8))
for name in ["P1", "P2"]:
    w = results.well(name)
    ax_oil.plot(w.time, w.oil_rate, label=name)
    ax_cut.plot(w.time, w.water_cut, label=name)
ax_oil.set(xlabel="time (days)", ylabel="oil rate (STB/day)", title="Oil rate")
ax_cut.set(xlabel="time (days)", ylabel="water cut", title="Water cut", ylim=(0, 1))
for ax in (ax_oil, ax_cut):
    ax.grid(alpha=0.3)
    ax.legend()
plt.show()

The two producers behave a little differently: they are at different distances from the injector, and the random
permeability gives each its own flow paths. Pressure at chosen blocks (here the three well blocks in the top layer) also
makes a line plot:

In [ ]:
top_layer_well_blocks = [grid.block_number(2, 5, 1), grid.block_number(19, 3, 1), grid.block_number(19, 8, 1)]
viz.plot_pressure_history(results, cells=top_layer_well_blocks)
plt.show()

## 4. 3-D data: pressure and water saturation

The pressure and saturation of **every block at every saved time** are in `results.pressure` and `results.sw`. A single time is
one row; `results.sw_3d(snapshot)` reshapes it into an array of shape `(layers, rows, columns)` = `(nz, ny, nx)`, so
`sw[k, j, i]` is the saturation of block `(i+1, j+1, k+1)`.

In [ ]:
sw_end = results.sw_3d(-1)                     # the last snapshot
p_end = results.pressure_3d(-1)
print("shape:", sw_end.shape, "= (nz, ny, nx)")
print(f"water saturation at the end: {sw_end.min():.2f} to {sw_end.max():.2f}")
print(f"pressure at the end:         {p_end.min():.0f} to {p_end.max():.0f} psi")
print(f"Sw of the block (i=2, j=5, k=3), next to the injector, bottom layer: {sw_end[2, 4, 1]:.2f}")

### 4.1 One time, every layer

`viz.plot_layers` shows all layers side by side on the same colour scale, here at the end of the run (day 1500). The triangles
are the wells: blue pointing up = injector, red pointing down = producer.

In [ ]:
viz.plot_layers(results, "sw")
plt.show()
viz.plot_layers(results, "pressure")
plt.show()

The water has swept much further in the bottom layer (600 md) than in the top layer (80 md): the **layering decides where the
water goes**. The pressure is highest around the injector, lowest around the producers.

### 4.2 A vertical slice

A **cross-section** cuts the model along a plane. Here we cut along the row `j = 5` (where the injector is), from left to right,
and look down into the layers. The thick line is the injector's completion.

In [ ]:
viz.plot_cross_section(results, "sw", plane="xz", index=5, vertical_exaggeration=8)
plt.show()

### 4.3 The whole model in 3-D

`viz.plot_voxels` draws every block as a coloured 3-D box. `cutaway=True` removes one corner so you can see inside.

In [ ]:
viz.plot_voxels(results, "sw", cutaway=True)
plt.show()

## 5. How things change over time

Every plot function takes `time=` (the nearest saved snapshot is used). Here is the **bottom layer**'s water saturation at four
times: the water front advances from the injector toward the producers.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6.2))
for ax, day in zip(axes.ravel(), [100, 400, 700, 1500]):
    viz.plot_layer(results, "sw", layer=3, time=day, ax=ax, colorbar=False, vmin=0.2, vmax=0.8)
fig.tight_layout()
fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0.2, 0.8), cmap="Blues"), ax=axes, label="water saturation")
plt.show()

Often the **change** is more telling than the state. The next cells compute the change since the start, block by block, with
plain NumPy subtraction: the water saturation gained and the pressure lost between day 0 and day 1500, in the bottom layer.

In [ ]:
layer = 2                                              # index 2 = layer 3, the bottom layer
d_sw = results.sw_3d(-1)[layer] - results.sw_3d(0)[layer]
d_p = results.pressure_3d(-1)[layer] - results.pressure_3d(0)[layer]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, data, cmap, label in ((ax1, d_sw, "Blues", "gain in Sw"), (ax2, -d_p, "Reds", "pressure drop (psi)")):
    mesh = ax.pcolormesh(grid.x_edges, grid.y_edges, data, cmap=cmap)
    ax.set(title=f"{label}, bottom layer", xlabel="x (ft)", ylabel="y (ft)", aspect="equal")
    fig.colorbar(mesh, ax=ax)
plt.show()

And a number you can quote: the **average** water saturation and pressure of the whole reservoir through time. Averaging
weights each block by its pore volume (volume × porosity), which here is the volume, since the porosity is the same everywhere.

In [ ]:
pore_volume = grid.volume * 0.2
avg_sw = results.sw @ pore_volume / pore_volume.sum()
avg_p = results.pressure @ pore_volume / pore_volume.sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))
ax1.plot(results.times, avg_sw)
ax1.set(xlabel="time (days)", ylabel="average Sw", title="Average water saturation")
ax2.plot(results.times, avg_p, color="tab:green")
ax2.set(xlabel="time (days)", ylabel="average pressure (psi)", title="Average pressure")
for ax in (ax1, ax2):
    ax.grid(alpha=0.3)
plt.show()

## 6. Movies

A movie shows the whole history in one picture. The three below are saved as GIF files into `examples/output/`
(an ignored folder) and shown here. Each movie has a fixed colour scale, so colours mean the same thing in every frame.

### 6.1 One layer

`viz.animate_layer` makes a time-lapse of one layer: the bottom layer's water front.

In [ ]:
from pathlib import Path

out = Path("output") if Path("output").is_dir() or Path.cwd().name == "examples" else Path("examples/output")
out.mkdir(parents=True, exist_ok=True)

gif = viz.animate_layer(results, out / "tutorial_layer3_sw.gif", field="sw", layer=3, fps=6, max_frames=20)
Image(filename=str(gif))

### 6.2 The whole 3-D model

`viz.animate_voxels` does the same for the 3-D model with a cut-away corner and a fixed camera.

In [ ]:
gif = viz.animate_voxels(results, out / "tutorial_model_sw.gif", field="sw", cutaway=True, fps=6, max_frames=20)
Image(filename=str(gif))

### 6.3 The average map

Reservoir engineers often look at the **average map**: all layers collapsed into one map, so that each `(x, y)` position has
one number. It answers "where in the plan view is the water, overall?" kusim has no ready-made function for this, but it takes
only a few lines, and it shows how to make your own movie from the results.

We average over the layers (`axis=0`), weighting by the pore volume of each block (a thick layer counts more than a thin
one). The movie shows the average water saturation and average pressure side by side.

In [ ]:
from matplotlib import animation

def average_map(results, field, snapshot):
    """Pore-volume weighted average over the layers of one saved field: a 2-D (ny, nx) array."""
    g = results.grid
    values = g.to_3d(getattr(results, field)[snapshot])
    weights = g.to_3d(g.volume * 0.2)                      # pore volume (porosity is 0.2 everywhere)
    return (values * weights).sum(axis=0) / weights.sum(axis=0)

frames = np.linspace(0, len(results.times) - 1, 20).astype(int)
limits = {
    "sw": (0.2, 0.8),
    "pressure": (results.pressure.min(), results.pressure.max()),
}
settings = {"sw": ("Blues", "average water saturation"), "pressure": ("viridis", "average pressure (psi)")}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, field in zip(axes, ["sw", "pressure"]):
    cmap, label = settings[field]
    fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(*limits[field]), cmap=cmap), ax=ax, label=label)

def draw(n):
    s = frames[n]
    for ax, field in zip(axes, ["sw", "pressure"]):
        cmap, label = settings[field]
        ax.clear()
        ax.pcolormesh(grid.x_edges, grid.y_edges, average_map(results, field, s), cmap=cmap,
                      vmin=limits[field][0], vmax=limits[field][1])
        ax.set(title=f"{label}, t = {results.times[s]:g} d", xlabel="x (ft)", ylabel="y (ft)", aspect="equal")
    fig.tight_layout()

movie = animation.FuncAnimation(fig, draw, frames=len(frames))
path = out / "tutorial_average_map.gif"
movie.save(path, writer=animation.PillowWriter(fps=6))
plt.close(fig)
Image(filename=str(path))

The average map hides the layering that the bottom-layer movie shows, but it is the picture to use for the field: it shows the
**sweep** of the plan area and the pressure support from the injector.

## 7. Keep your results

Everything can be written to files for other programs (a spreadsheet, ParaView for 3-D, or another script):

In [ ]:
results.to_csv(out / "tutorial_results")                 # wells.csv, perforations.csv, pressure.csv, sw.csv
viz.export_vtk(results, out / "tutorial_paraview")       # open the .pvd file in ParaView
print(sorted(p.name for p in (out / "tutorial_results").iterdir()))

For an interactive 3-D view with a time slider (needs `plotly`, in the `[dev]` install), use
`viz.save_html(viz.interactive_3d(results, "sw", cutaway=True, opacity=0.6), "model_sw.html")` and open the file in a browser.

## 8. The same model from an ECLIPSE-style deck

So far we built the model in Python. Many engineers already have their models as an **ECLIPSE-style deck**: a text file of
keywords. kusim reads a deck (the part of ECLIPSE that kusim can represent: oil-water, oilfield units, Cartesian grid) and turns
it into the same `ReservoirModel`. The file `eclipse/sample_tutorial_case.DATA` describes the model of this tutorial.

In [ ]:
deck = Path("eclipse/sample_tutorial_case.DATA") if Path("eclipse").is_dir() else Path("examples/eclipse/sample_tutorial_case.DATA")
print(deck.read_text())

Each part of the deck matches a step of Section 1:

| Deck keywords | What they set | Tutorial step |
|---|---|---|
| `DIMENS`, `DXV`, `DYV`, `DZ`, `TOPS` | the grid | 1.1 |
| `PORO`, `PERMX`, `COPY`/`MULTIPLY` (PERMY, PERMZ) | porosity and permeability | 1.2 |
| `PVDO`, `PVTW`, `SWOF`, `DENSITY`, `ROCK` | fluids and rock | 1.3 |
| `PRESSURE`, `SWAT` | the initial state (uniform, as in 1.4) | 1.4 |
| `WELSPECS`, `COMPDAT`, `WCONINJE`, `WCONPROD` | the wells; the connections use `i, j, k` | 1.5 |
| `TSTEP` | `300*5` = 300 steps of 5 days | 1.6 |

Notice what you did **not** have to do: count block numbers. `COMPDAT` says `k1 = 1` to `k2 = 3`: the well is open in layers 1 to 3.
Ranges like `200*15` mean "200 copies of 15" (one layer has 200 blocks). The `SUMMARY` section only lists what ECLIPSE would
write out; kusim ignores it, and tells you so.

Read the deck:

In [ ]:
from kusim import read_deck

imported = read_deck(deck)
deck_model = imported.model

print("deck well names -> kusim names:", imported.wells)
print("keywords read but not used by kusim:", [item["keyword"] for item in imported.ignored])
print("warnings:", imported.warnings or "none")
print("control changes after day 0:", imported.schedule)
print(f"grid {deck_model.grid.nx} x {deck_model.grid.ny} x {deck_model.grid.nz}, dt = {deck_model.dt:g} d, t_end = {deck_model.t_end:g} d")

The reader is **strict**: a keyword that kusim cannot honour (gas, corner-point grids, ...) stops the import with an error that
names the file and line, instead of being skipped silently. The list `ignored` shows keywords that are accepted but have no effect.

This deck has no well-control changes after day 0 (`control changes: 0 events`), so a plain `run()` is enough. If a deck changes
rates later (another `WCONINJE` after a `TSTEP`), those changes come back in `imported.schedule`, and you would advance the
simulator with it (see `docs/eclipse-deck.md`).

In [ ]:
print("runnable:", check_model(deck_model)["runnable"])
deck_results = Simulator(deck_model, save_every=4).run()

viz.plot_wells(deck_results)
plt.show()

In [ ]:
import plotly
fig = viz.interactive_3d(results, "sw",cutaway=False, opacity=0.9)
fig

Everything in Sections 3 to 7 works on `deck_results` as well, because it is the same kind of object. Compare the two runs: the
oil rates are close, but breakthrough comes **later** in the deck run (about day 1090 against 800).

In [ ]:
fig, (ax_oil, ax_cut) = plt.subplots(1, 2, figsize=(11, 3.8))
for label, res in (("built in Python", results), ("read from the deck", deck_results)):
    w = res.well("P1")
    ax_oil.plot(w.time, w.oil_rate, label=label)
    ax_cut.plot(w.time, w.water_cut, label=label)
ax_oil.set(xlabel="time (days)", ylabel="oil rate (STB/day)", title="P1 oil rate")
ax_cut.set(xlabel="time (days)", ylabel="water cut", title="P1 water cut", ylim=(0, 1))
for ax in (ax_oil, ax_cut):
    ax.grid(alpha=0.3)
    ax.legend()
plt.show()

**Why the two differ.** The deck is the same reservoir, but its tables are not identical to the ones we typed in Python: the deck's
`SWOF` has a **capillary pressure** column (8 psi at the connate saturation, falling to 0) and a different relative permeability
shape, whereas our Python table used none, and `PVTW` gives the water's formation volume factor at one pressure. A different
input is a different answer; that is the point of keeping the model in one place.

**Two things to remember when writing a deck for kusim**

* Every record ends with `/`, including `DIMENS`. Without it the reader runs on into the next keyword, which gives a confusing
  error ("FIELD is not declared").
* The initial state here is `PRESSURE` and `SWAT` with one value for every block (`600*3000`, `600*0.2`: 20 x 10 x 3 = 600
  blocks). Real decks usually use `EQUIL`, which sets pressure from a datum depth and follows the fluid gradient.

You can also run the deck from the command line (`kusim import-deck eclipse/sample_tutorial_case.DATA -o model.json`),
which writes the same model as a JSON file, ready for `kusim run`.

## 9. Try it yourself

Change one thing, **run all cells again** (in Sections 1 to 7; edit the deck for Section 8), and compare. Each is a one-line edit:

1. **Inject more.** Change `1200.0` in the injector to `600.0`. What happens to the pressure and to the water breakthrough time?
2. **Remove the layering.** Set `layer_permeability = np.array([300.0, 300.0, 300.0])`. Is the water front still faster in the
   bottom layer? (Hint: the random variation is still there, so it is not perfectly even.)
3. **Move a well.** Put producer 2 at `(i=19, j=10)` instead: `grid.column_blocks(19, 10)`. Does the water cut curve change?
4. **Complete the injector in one layer only**: `grid.block_number(2, 5, 3)` instead of the whole column. Which layer floods now?
5. **A longer test.** Set `t_end=3000.0`. Does the water cut reach 100 %? What stops it?

## Where next

* `docs/json-format.md` and `docs/units-and-assumptions.md`: how to save a model as a JSON file and run it from the command line
  (`kusim run`), and what the simulator assumes.
* `examples/03_visualization.py`: every viewer in one script, including the ParaView export.
* `docs/history-matching.md`: adjusting the model to match measured production data.